In [1]:
%pip install -q -U transformers datasets peft trl accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch, platform

print("Machine:", platform.platform())
print("CUDA available:", torch.cuda.is_available())
print("Apple MPS available:", torch.backends.mps.is_available())

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Machine: macOS-15.5-arm64-arm-64bit-Mach-O
CUDA available: False
Apple MPS available: True
Using device: mps


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-0.5B-Instruct"   # lighter option: "HuggingFaceTB/SmolLM2-360M-Instruct"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)
model.to(device)
print("Model loaded on", device)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
W1116 08:44:39.659000 13423 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on mps


In [4]:
from datasets import Dataset

SYS = "You are the assistant of TechNest Computers."
qa = [
("What are your opening hours?", "TechNest Computers is open Monday to Saturday, 9:00 AM to 7:00 PM. We are closed on Sundays."),
("Where is your shop located?", "We are at No. 45, Main Street, Kegalle, next to the central bus stand."),
("Do you sell laptops?", "Yes, we sell new laptops from HP, Dell, Lenovo, Asus and Acer."),
("Do you sell second-hand laptops?", "Yes, we sell certified refurbished laptops with a 6-month warranty."),
("What is the warranty on new laptops?", "New laptops come with a 1-year manufacturer warranty, extendable to 3 years."),
("Do you repair laptops?", "Yes, our service centre repairs screens, keyboards, batteries and motherboards."),
("How long does a laptop repair take?", "Most repairs take 2 to 5 working days, depending on spare part availability."),
("Is there a diagnostic fee?", "The diagnostic fee is Rs. 1,500, which is waived if you proceed with the repair."),
("Can you upgrade my laptop RAM?", "Yes. We upgrade RAM and install SSDs. Most upgrades are done the same day."),
("How much is a 16GB DDR4 RAM stick?", "A 16GB DDR4 RAM stick costs about Rs. 18,000 including installation."),
("Do you sell SSDs?", "Yes, we stock 256GB, 512GB and 1TB NVMe and SATA SSDs from Samsung and Kingston."),
("Do you build custom desktop PCs?", "Yes, we build custom gaming and office PCs. Assembly is free if you buy all parts from us."),
("How long does a custom PC build take?", "A custom PC is usually ready within 3 working days after all parts are available."),
("Which graphics cards do you sell?", "We stock NVIDIA RTX 3050, 4060 and 4070 and AMD RX 6600 cards."),
("Do you sell monitors?", "Yes, we sell 22 to 32 inch monitors from Dell, LG and Samsung, including 144Hz gaming models."),
("Do you sell printers?", "Yes, we sell inkjet and laser printers from Canon, Epson and HP, plus ink and toner."),
("Can I refill printer cartridges here?", "Yes, we refill ink and toner cartridges. Inkjet refills take about 30 minutes."),
("Do you accept credit cards?", "Yes, we accept Visa and Mastercard, as well as cash and bank transfers."),
("Do you offer installment plans?", "Yes, we offer 0% installment plans for 6 or 12 months on selected bank cards."),
("Do you deliver?", "We deliver island-wide in 2 to 3 working days. Delivery is free for orders above Rs. 100,000."),
("What is your return policy?", "Unused items in original packaging can be returned within 7 days with the receipt."),
("Can I return a laptop after opening it?", "Opened laptops can only be exchanged if they have a manufacturing fault, within 7 days."),
("Do you install Windows?", "Yes, we install Windows 10 and 11 and Microsoft Office for Rs. 3,000."),
("Do you remove viruses?", "Yes, our virus removal and system clean-up service costs Rs. 2,500."),
("Can you recover deleted data?", "We attempt data recovery from hard disks and USB drives. Fees start at Rs. 5,000."),
("Do you sell keyboards and mice?", "Yes, we sell wired and wireless keyboards and mice from Logitech, A4Tech and Redragon."),
("Do you sell UPS units?", "Yes, we sell 650VA to 2000VA UPS units with a 1-year warranty."),
("Do you sell Wi-Fi routers?", "Yes, we sell TP-Link and Tenda routers and can set up your home network."),
("Do you offer student discounts?", "Yes, students get a 5% discount on laptops and accessories with a valid student ID."),
("How can I contact the shop?", "Call 081-222-3344 or WhatsApp 077-123-4567, or email info@technest.lk."),
]

ds = Dataset.from_list([
    {"messages": [
        {"role": "system", "content": SYS},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]} for q, a in qa
])
print("Training examples:", len(ds))

Training examples: 30


In [5]:
test_qs = [
    "What time do you open on Saturday?",         # paraphrase of training
    "How long is the warranty on a new laptop?",  # paraphrase of training
    "Can you fix my broken laptop screen?",       # paraphrase of training
    "Do you sell webcams?",                       # NOT in training data
    "Can I pay with PayPal?",                     # NOT in training data
    "Do you have a branch in Colombo?",           # NOT in training data
]

def ask(m, q):
    msgs = [{"role": "system", "content": SYS}, {"role": "user", "content": q}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(m.device)
    out = m.generate(**ids, max_new_tokens=80, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)

base_answers = [ask(model, q) for q in test_qs]
for q, a in zip(test_qs, base_answers):
    print("Q:", q, "\nBASE:", a, "\n")

Q: What time do you open on Saturday? 
BASE: As an AI language model, I don't have a physical presence or schedule like humans do. However, I'm always ready to assist with any questions or tasks you may have! Please let me know how I can help you today. 

Q: How long is the warranty on a new laptop? 
BASE: The warranty period for a new laptop can vary depending on the brand and model, but generally, it ranges from 1 to 3 years or more in some cases.

For example:

- **Apple**: The standard warranty for an Apple MacBook Air is typically 2 years (or 5 years if you purchase a refurbished model).
- **Lenovo**: Lenovo laptops come with a 2-year warranty 

Q: Can you fix my broken laptop screen? 
BASE: I'm sorry to hear that your laptop screen is broken! Here are some steps you can take to try and fix it:

1. Check if the screen is damaged: If the screen is cracked or has a scratch, you may need to replace it.

2. Try using a different monitor: If the screen is still working on one computer,

In [6]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

args = SFTConfig(
    output_dir="out",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    fp16=False,
    bf16=False,
    report_to="none",
    save_strategy="no",
)

trainer = SFTTrainer(model=model, args=args, train_dataset=ds,
                     peft_config=lora, processing_class=tok)
trainer.train()

/opt/anaconda3/lib/python3.13/site-packages/transformers/training_args.py:1541: FutureWarning: `gradient_checkpointing` is deprecated and will be removed in v6. Use `activation_checkpointing` instead.
  warnings.warn(f"`{old}` is deprecated and will be removed in v6. Use `{new}` instead.", FutureWarning)
/opt/anaconda3/lib/python3.13/site-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/30 [00:00<?, ? examples/s]

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
5,2.925130
10,1.222847
15,0.873685
20,0.650373
25,0.466772
30,0.401052
35,0.317232
40,0.287273


TrainOutput(global_step=40, training_loss=0.8930456668138504, metrics={'train_runtime': 30.1041, 'train_samples_per_second': 4.983, 'train_steps_per_second': 1.329, 'total_flos': 18802720504320.0, 'train_loss': 0.8930456668138504, 'epoch': 5.0})

In [7]:
import pandas as pd

ft_answers = [ask(trainer.model, q) for q in test_qs]

cmp = pd.DataFrame({"Question": test_qs, "Base model": base_answers, "Fine-tuned": ft_answers})
pd.set_option("display.max_colwidth", None)
display(cmp)

cmp.to_csv("comparison.csv", index=False)
trainer.model.save_pretrained("technest-lora")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


,Question,Base model,Fine-tuned
0,What time do you open on Saturday?,"As an AI language model, I don't have a physical presence or schedule like humans do. However, I'm always ready to assist with any questions or tasks you may have! Please let me know how I can help you today.",Wesystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem
1,How long is the warranty on a new laptop?,"The warranty period for a new laptop can vary depending on the brand and model, but generally, it ranges from 1 to 3 years or more in some cases.\n\nFor example:\n\n- **Apple**: The standard warranty for an Apple MacBook Air is typically 2 years (or 5 years if you purchase a refurbished model).\n- **Lenovo**: Lenovo laptops come with a 2-year warranty",Newsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem
2,Can you fix my broken laptop screen?,"I'm sorry to hear that your laptop screen is broken! Here are some steps you can take to try and fix it:\n\n1. Check if the screen is damaged: If the screen is cracked or has a scratch, you may need to replace it.\n\n2. Try using a different monitor: If the screen is still working on one computer, you can try using a different monitor to see if the",Yessystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem
3,Do you sell webcams?,"Yes, I can help you with that! What would you like to know about selling webcams?",Yessystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem
4,Can I pay with PayPal?,"Yes, you can pay with PayPal using a credit card or bank transfer. Here's how:\n\n1. **Credit Card**: You'll need to enter your credit card information into PayPal and then proceed with the payment.\n\n2. **Bank Transfer**: If you're paying via a bank transfer, you'll need to provide the recipient's bank account details along with the amount to be transferred.\n\n3. **",Yessystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem
5,Do you have a branch in Colombo